This will be the main top x CF file

In [ ]:
import xarray as xr
import numpy as np
import geopandas as gpd
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import shapely.vectorized as sv
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def average_capacity_factors(filename):
    '''
    Function to read in a .nc file and calculate the average capacity factors over time.
    '''
    ds = xr.open_dataset(filename)

    cf_avg = ds.mean(dim='time')

    return cf_avg

def get_top_60_percent_cfs(avg_capacity_factors):
    '''
    Function to get the top 60% of capacity factors.
    '''
    cfs_60 = avg_capacity_factors.where(avg_capacity_factors > avg_capacity_factors.quantile(0.01), -1)

    # Assuming 'variable_name' is the name of the variable in the Dataset
    num_cfs_60 = cfs_60['capacity factor'].where(cfs_60['capacity factor'] != -1).notnull().sum().item()
    print(f"Number of grid cells in the top 60%: {num_cfs_60}/{cfs_60['capacity factor'].size}")

    return cfs_60

In [ ]:
def get_time_series_top_60_percent_cfs(filename, cfs_60, cf_type):
    '''
    Function to get the time series of the top 60% of capacity factors.
    '''
    ds = xr.open_dataset(filename)

    # Save average over all grid cells to csv file
    ds_avg = ds.mean(dim=['x', 'y'])
    ds_avg_df = ds_avg.to_dataframe()
    ds_avg_df = ds_avg_df.rename(columns={'capacity factor': f'{cf_type}_cf_avg'})
    year = filename.split('_')[-1].split('.')[0]
    # ds_avg_df.to_csv(f'CONUS_{cf_type}_CF_avg_all_{year}.csv')

    # Get time series of top 60% grid cells
    cfs_60_ts = ds.where(cfs_60 > 0)

    # Average over all grid cells
    cfs_60_ts = cfs_60_ts.mean(dim=['x', 'y'])

    # Write to csv file
    cfs_60_df = cfs_60_ts.to_dataframe()

    # Drop columns execpt for time and capacity factor
    cfs_60_df = cfs_60_df.drop(columns=['quantile'])
    # Rename columns
    cfs_60_df = cfs_60_df.rename(columns={'capacity factor': f'{cf_type}_cf'})
    
    # Write to csv file
    year = filename.split('_')[-1].split('.')[0]
    cfs_60_df.to_csv(f'CONUS_{cf_type}_CF_{year}.csv')

In [ ]:
def plot_capacity_factor(cf, cap_fac_type):
    from shapely.validation import make_valid

    miso = gpd.read_file("/Users/mshaaban/clab_underground_th_storage/data_folder/wind_solar_capacity_factors_MISO/miso_shapefile.geojson")
    if miso.crs is None or miso.crs.to_epsg() != 4326:
        miso = miso.to_crs(epsg=4326)
    #miso_geom = miso.geometry.union_all()

        # Fix invalid geometries
    miso["geometry"] = miso["geometry"].apply(make_valid)
    miso_geom = miso.geometry.union_all()

    lon, lat = np.meshgrid(cf['x'], cf['y'])
    mask = sv.contains(miso_geom, lon, lat)
    masked_dataset = cf.where(mask)

    aspect_ratio = (cf['x'].max()-cf['x'].min())/(cf['y'].max()-cf['y'].min())
    masked_dataset["capacity factor"].plot(aspect=aspect_ratio, size=5, vmin=0.)
    plt.title(f'{cap_fac_type} MISO average')
    plt.savefig(f'capacity_factor_MISO_{cap_fac_type.replace(" ","").replace("%","")}.jpg')
    print(masked_dataset["capacity factor"].where(masked_dataset["capacity factor"] > 0).mean().values)

In [ ]:
def get_60_percent_cf_all(cf_type):
    """
    Get the top 60% capacity factors for all years
    """

    cf_avg_2020 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2020.nc')
    cf_avg_2021 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2021.nc')
    cf_avg_2022 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2022.nc')
    cf_avg_2024 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2024.nc')
    cf_avg_2023 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2023.nc')

    cf_avg_all = (cf_avg_2020 + cf_avg_2021 + cf_avg_2022 + cf_avg_2023 + cf_avg_2024)/5

    # Plot 
    plot_capacity_factor(cf_avg_all, cf_type)
    cf_all_60 = get_top_60_percent_cfs(cf_avg_all)
    plot_capacity_factor(cf_all_60, cf_type+" top 60%")

    cfs = []
    for year in range(2020, 2025):

        cf_ts = xr.open_dataset(f'miso_{cf_type}_CF_timeseries_{year}.nc')

        # Only keep the capacity factors that are non-zero in cf_all_60
        cf_ts = cf_ts.where(cf_all_60 > -1)

        # Average over all grid cells
        cf_ts = cf_ts.mean(dim=['x', 'y'])

        cfs.append(cf_ts)

    cf_ts_all = xr.concat(cfs, dim='time')
    # Drop quantile dimension
    cf_ts_all = cf_ts_all.drop('quantile')
    # Rename capacity factor dimension
    cf_ts_all = cf_ts_all.rename({'capacity factor': f'{cf_type}_cf'})
    # Write to csv file
    cf_ts_all_df = cf_ts_all.to_dataframe()
    cf_ts_all_df.to_csv(f'MISO_{cf_type}_CF_2020_2024.csv')


In [ ]:
get_60_percent_cf_all('solar')
get_60_percent_cf_all('wind')

In [ ]:
def read_csv_file(filename):
    """
    Read csv file
    """
    df = pd.read_csv(filename)
    # Print mean of columm 1
    print(df.iloc[:,1].mean())

# Source for reported capacity factors:
# https://www.eia.gov/electricity/monthly/epm_table_grapher.php?t=table_6_07_b
print("Solar")
print("Reported 5 year average of capacity factors")
print((20.38+22.26+23.03+22.83+21.81)/5./100.)
print("Mean capacity factor of top 60% of grid cells")
read_csv_file('MISO_solar_CF_2020_2024.csv')

print("\n")
print("Wind")
print("Reported 5 year average of capacity factors")
print((31.66+32.13+36.65+31.76+33.15)/5./100.)
print("Mean capacity factor of top 60% of grid cells")
read_csv_file('MISO_wind_CF_2020_2024.csv')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.validation import make_valid
import cartopy.io.shapereader as shpreader

def plot_miso_boundary(geojson_path, outfile="miso_boundary.png", show_states=True):
    miso = gpd.read_file(geojson_path)
    if miso.crs is None or miso.crs.to_epsg() != 4326:
        miso = miso.to_crs(epsg=4326)
    miso["geometry"] = miso["geometry"].apply(make_valid)

    fig, ax = plt.subplots(figsize=(7, 7))

    # Optional: US state outlines for context
    if show_states:
        states = gpd.read_file(shpreader.natural_earth(
            resolution="50m", category="cultural", name="admin_1_states_provinces"))
        states = states[states["admin"] == "United States of America"]
        states.boundary.plot(ax=ax, linewidth=0.4, color="lightgray")

    miso.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2)

    # Zoom to the MISO extent with a small pad
    minx, miny, maxx, maxy = miso.total_bounds
    pad = 1.0
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("MISO boundary")
    fig.tight_layout()
    fig.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()

plot_miso_boundary(
    "/Users/mshaaban/clab_underground_th_storage/data_folder/"
    "wind_solar_capacity_factors_MISO/miso_shapefile.geojson"
)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.validation import make_valid

DIR = Path("/Users/mshaaban/clab_underground_th_storage/data_folder/wind_solar_capacity_factors_Map_with_bounderies")   # folder containing the 6 files
files = ["1", "4", "6", "27", "35", "8910"]

# Load, combine, clean
zones = pd.concat([gpd.read_file(DIR / f"{f}.geojson") for f in files], ignore_index=True)
zones = zones.set_crs(4326, allow_override=True)
zones["geometry"] = zones.geometry.apply(make_valid)
zones = zones[zones["Shape__Area"] > 1e6]            # drop sliver polygons
zones = zones.dissolve(by="Zone", as_index=False)    # one row per LRZ
zones["n"] = zones["Zone"].str.extract(r"(\d+)").astype(int)
zones = zones.sort_values("n").reset_index(drop=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.cm.tab10
for i, row in zones.iterrows():
    gpd.GeoSeries([row.geometry], crs=4326).plot(
        ax=ax, facecolor=cmap(i % 10)[:3] + (0.35,), edgecolor="black", linewidth=1.0)
    pt = row.geometry.representative_point()
    ax.annotate(row["Zone"], (pt.x, pt.y), ha="center", va="center", fontsize=8, weight="bold")

minx, miny, maxx, maxy = zones.total_bounds
ax.set_xlim(minx - 0.5, maxx + 0.5)
ax.set_ylim(miny - 0.5, maxy + 0.5)
ax.set_aspect("equal")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("MISO Local Resource Zones")
fig.tight_layout()
fig.savefig("miso_lrz_boundaries.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def load_zones(dir_path, files=("1", "4", "6", "27", "35", "8910")):
    z = pd.concat([gpd.read_file(Path(dir_path) / f"{f}.geojson") for f in files], ignore_index=True)
    z = z.set_crs(4326, allow_override=True)
    z["geometry"] = z.geometry.apply(make_valid)
    z = z[z["Shape__Area"] > 1e6].dissolve(by="Zone", as_index=False)
    z["n"] = z["Zone"].str.extract(r"(\d+)").astype(int)
    return z.sort_values("n").reset_index(drop=True)

def overlay_zones(ax, zones, labels=True, lw=0.9):
    zones.boundary.plot(ax=ax, color="black", linewidth=lw)
    if labels:
        for _, r in zones.iterrows():
            pt = r.geometry.representative_point()
            ax.annotate(r["Zone"], (pt.x, pt.y), ha="center", va="center",
                        fontsize=7, weight="bold", color="black",
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.6))

In [ ]:
def plot_capacity_factor(cf, cap_fac_type, zones, ax=None, labels=True, save=True):
    # Mask to the union of the zones so the data and outlines share one footprint
    zones_geom = zones.geometry.union_all()
    lon, lat = np.meshgrid(cf['x'], cf['y'])
    mask = sv.contains(zones_geom, lon, lat)
    da = cf.where(mask)["capacity factor"]
    da = da.where(da >= 0)

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 5))

    da.plot(ax=ax, vmin=0, cbar_kwargs={"label": "Capacity factor", "shrink": 0.8})
    overlay_zones(ax, zones, labels=labels)

    minx, miny, maxx, maxy = zones.total_bounds
    ax.set_xlim(minx - 0.5, maxx + 0.5)
    ax.set_ylim(miny - 0.5, maxy + 0.5)
    ax.set_aspect("equal")
    ax.set_title(f"{cap_fac_type} MISO average")

    if standalone and save:
        plt.savefig(f'capacity_factor_MISO_{cap_fac_type.replace(" ","").replace("%","")}.jpg',
                    dpi=300, bbox_inches="tight")
    print(da.where(da > 0).mean().values)

In [ ]:
def get_60_percent_cf_all(cf_type, zones):
    ...
    plot_capacity_factor(cf_avg_all, cf_type, zones)
    cf_all_60 = get_top_60_percent_cfs(cf_avg_all)
    plot_capacity_factor(cf_all_60, cf_type + " top 60%", zones)
    ...
    return cf_avg_all, cf_all_60

In [ ]:
def get_60_percent_cf_all(cf_type, zones, plot=False):
    """
    Get the top 60% capacity factors for all years
    """
    cf_avg_2020 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2020.nc')
    cf_avg_2021 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2021.nc')
    cf_avg_2022 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2022.nc')
    cf_avg_2023 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2023.nc')
    cf_avg_2024 = average_capacity_factors(f'miso_{cf_type}_CF_timeseries_2024.nc')

    cf_avg_all = (cf_avg_2020 + cf_avg_2021 + cf_avg_2022 + cf_avg_2023 + cf_avg_2024) / 5

    cf_all_60 = get_top_60_percent_cfs(cf_avg_all)

    if plot:  # standalone figures with zone overlay
        plot_capacity_factor(cf_avg_all, cf_type, zones)
        plot_capacity_factor(cf_all_60, cf_type + " top 60%", zones)

    cfs = []
    for year in range(2020, 2025):
        cf_ts = xr.open_dataset(f'miso_{cf_type}_CF_timeseries_{year}.nc')

        # Only keep the capacity factors that are non-zero in cf_all_60
        cf_ts = cf_ts.where(cf_all_60 > -1)

        # Average over all grid cells
        cf_ts = cf_ts.mean(dim=['x', 'y'])
        cfs.append(cf_ts)

    cf_ts_all = xr.concat(cfs, dim='time')
    cf_ts_all = cf_ts_all.drop('quantile')
    cf_ts_all = cf_ts_all.rename({'capacity factor': f'{cf_type}_cf'})
    cf_ts_all.to_dataframe().to_csv(f'MISO_{cf_type}_CF_2020_2024.csv')

    return cf_avg_all, cf_all_60

In [ ]:
from shapely.geometry import MultiPolygon

def polys_only(g):
    if g.geom_type in ("Polygon", "MultiPolygon"):
        return g
    polys = []
    for p in g.geoms:
        if p.geom_type == "Polygon":
            polys.append(p)
        elif p.geom_type == "MultiPolygon":
            polys.extend(p.geoms)
    return MultiPolygon(polys)

def load_zones(dir_path, files=("1", "4", "6", "27", "35", "8910")):
    z = pd.concat([gpd.read_file(Path(dir_path) / f"{f}.geojson") for f in files], ignore_index=True)
    z = z.set_crs(4326, allow_override=True)
    z["geometry"] = z.geometry.apply(make_valid)
    z = z[z["Shape__Area"] > 1e6].dissolve(by="Zone", as_index=False)
    z["geometry"] = z.geometry.apply(polys_only)          # <-- new
    z["n"] = z["Zone"].str.extract(r"(\d+)").astype(int)
    return z.sort_values("n").reset_index(drop=True)

In [ ]:
print(zones.geom_type.unique())   # should be only Polygon / MultiPolygon

In [ ]:
zones["geometry"] = zones.geometry.apply(polys_only)
print(zones.geom_type.unique())   # should now be only MultiPolygon / Polygon

In [ ]:
bad = zones[zones.geom_type == "GeometryCollection"]
for _, r in bad.iterrows():
    print(r["Zone"], [g.geom_type for g in r.geometry.geoms])

In [ ]:
zones = load_zones("/Users/mshaaban/clab_underground_th_storage/data_folder/wind_solar_capacity_factors_Map_with_bounderies")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, cf_type in enumerate(["solar", "wind"]):
    cf_avg_all, cf_all_60 = get_60_percent_cf_all(cf_type, zones)
    plot_capacity_factor(cf_avg_all, cf_type, zones, ax=axes[row, 0])
    plot_capacity_factor(cf_all_60, f"{cf_type} top 60%", zones, ax=axes[row, 1])

fig.tight_layout()
fig.savefig("capacity_factor_MISO_4panel_zones.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
zones = load_zones("/Users/mshaaban/clab_underground_th_storage/data_folder/wind_solar_capacity_factors_Map_with_bounderies")

for cf_type in ["Solar", "Wind"]:
    cf_avg_all, cf_all_60 = get_60_percent_cf_all(cf_type, zones)
    plot_capacity_factor(cf_avg_all, cf_type, zones)
    plt.show()
    plot_capacity_factor(cf_all_60, f"{cf_type} Avg CF MISO 2020-2024", zones)
    plt.show()